In [1]:
from  langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv

# This function will load all the variables from the .env file and will 
# make them available in the os.environ dictionary (env variables)
load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

Bro API KEY Variable exists


CHAIN WITH Parallel Chains

In [2]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")])

In [3]:
# TASK - 2 [LLM]

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

In [4]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [5]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [6]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

Chain 1

In [9]:
def linked_chain(text:dict):

    text = text["text"]
    linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])
    
  

    chain_linkedin = linkedin_prompt | llm_openai | str_parser

    result = chain_linkedin.invoke(text)

    return result

linked_chain = RunnableLambda(insta_chain)

Chain 2

In [7]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

Final Chain

In [10]:
final_chain = ( 
                    prompt_template | 
                    llm_openai | 
                    str_parser | 
                    dictionary_maker_runnable |
                    RunnableParallel(branches = {"linkedin": linked_chain, "instagram": insta_chain_runnable})
)

In [11]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'KGF: Chapter 1 (2018) — a raw, stylish tale of ambition, power and leadership. 🔥\n\nDirected by Prashanth Neel and anchored by Yash as Rocky, the film follows a poor young man who promises his dying mother to seize power and wealth. Sent to the brutal, gold-rich Kolar Gold Fields to assassinate a tyrant, Rocky becomes the reluctant savior of oppressed miners — sparking a violent struggle for control. The result is high-energy, visually striking cinema that ends with Rocky consolidating power and setting the stage for the next chapter. 🏆⛏️\n\nIf you haven’t seen it yet, it’s a must-watch for fans of bold storytelling and larger-than-life protagonists. What did you take away from Rocky’s rise — ambition, leadership, or something darker? Share your thoughts. 👇\n\n#KGF #KGFChapter1 #Yash #PrashanthNeel #IndianCinema #KannadaCinema #FilmRecommendation #LeadershipInFilm #MustWatch',
  'instagram': 'Rocky. Power. Gold. Legacy. 💥🔥\n\nFrom Mumbai’s streets to the brut

Chain as a Runnable

In [14]:
# TASK - 1 [Beautify Function]

def beautify(final_response:dict)-> dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


# TASK - 2 [Final Chain]

# final_chain 


# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Vadh")

{'linkedin': 'When patience runs out, what do you become? 🎬🖤\n\nVadh is a tense, unflinching crime drama about a mild‑mannered retired government employee whose quiet life is eroded by a local thug and an uncaring system. Pushed beyond his limits by thefts, humiliation and threats, he snaps — and a single violent act sets off a chain of consequences that forces him to face the moral and legal cost of survival.\n\nA haunting story of a man’s transformation from timid citizen to a person driven by rage, Vadh explores justice, social breakdown and the fragile limits of patience.\n\nWould you stay silent — or fight back? Share your thoughts below. ⬇️\n\n#Vadh #CrimeDrama #MoralDilemma #Thriller #Justice #FilmRecommendation #MustWatch',
 'instagram': 'How far can patience stretch before it snaps? 🔥\n\nVadh follows a mild‑mannered, retired government employee whose peaceful life in a small town is slowly destroyed by a local criminal and indifferent authorities. After thefts, humiliation and